In [260]:
import pandas as pd
import numpy as np
from pyomo.environ import ConcreteModel, Var, NonNegativeIntegers, SolverFactory, Objective, maximize,Param,Set,Constraint,value,Binary
import matplotlib.pyplot as plt
from pathlib import Path

In [9]:
gold_dir = Path('../data/gold')
dim_produto = pd.read_parquet(gold_dir / 'dim_produto.parquet')
dim_consumidor = pd.read_parquet(gold_dir / 'dim_consumidor.parquet')
dim_tempo = pd.read_parquet(gold_dir / 'dim_tempo.parquet')
fact_credito = pd.read_parquet(gold_dir / 'gold_credito.parquet')
df_predicao = pd.read_parquet(gold_dir / 'predicao_glm.parquet')
df_predicao.columns = ['sk_consumidor', 'sk_produto', 'taxa_conversao']

In [212]:
P = dim_produto['sk_produto']
S = dim_consumidor['sk_consumidor']
max_ofertas_por_produto = 0.25
min_diversidade = 10

In [13]:
df_parametros = fact_credito.groupby(['sk_produto', 'sk_consumidor']).agg({'taxa_conversao': 'mean','possui_oferta':'sum','conversao_efetiva':'sum','usuarios_elegiveis':'sum','receita_gerada':'sum'}).reset_index()

In [120]:
df_taxa_conversao = df_parametros[['sk_produto','sk_consumidor','taxa_conversao']]
df_taxa_conversao = pd.concat([df_taxa_conversao, df_predicao], axis=0)
df_taxa_conversao.replace(0,0.0001,inplace=True)

In [15]:
df_ticket_medio = df_parametros.groupby(['sk_produto']).agg({'receita_gerada': 'sum','conversao_efetiva': 'sum'}).reset_index()
df_ticket_medio['ticket_medio'] = df_ticket_medio['receita_gerada'] / df_ticket_medio['conversao_efetiva']
df_ticket_medio = df_ticket_medio[['sk_produto', 'ticket_medio']]
df_ticket_medio

,sk_produto,ticket_medio
0,1,1.263292
1,2,1.160574
2,3,3.249895
3,4,16.469095
4,5,9.034290
5,6,1.229703
6,7,123.000000


In [76]:
df_usuarios_elegiveis = df_parametros.groupby(['sk_consumidor']).agg({'usuarios_elegiveis': 'sum'}).reset_index()
df_usuarios_elegiveis

,sk_consumidor,usuarios_elegiveis
0,1,13352.0
1,2,272324.0
2,3,76423.0
3,4,12735.0
4,5,64743.0
...,...,...
1310,1311,2.0
1311,1312,1.0
1312,1313,2.0
1313,1314,5.0


In [22]:
total_ofertas = df_parametros.agg({'possui_oferta': 'sum'}).reset_index()[0][0]
total_ofertas

18132681.0

In [121]:
#converter para tupledict do pyomo
taxa_conversao_is = df_taxa_conversao.set_index(['sk_produto', 'sk_consumidor'])['taxa_conversao'].to_dict()
ticket_medio_i = df_ticket_medio.set_index('sk_produto')['ticket_medio'].to_dict()
usuarios_elegiveis_s = df_usuarios_elegiveis.set_index('sk_consumidor')['usuarios_elegiveis'].to_dict()

In [128]:
model.P.display()

P : Size=1, Index=None, Ordered=Insertion
    Key  : Dimen : Domain : Size : Members
    None :     1 :    Any :    7 : {1, 2, 3, 4, 5, 6, 7}


In [216]:
dim_consumidor.shape

(1315, 8)

In [345]:
# ====================== DADOS (você já tem) ======================
# df_taxa_conversao   → sk_produto, sk_consumidor, taxa_conversao
# df_ticket_medio     → sk_produto, ticket_medio
# df_usuarios_elegiveis → sk_consumidor, usuarios_elegiveis
# total_ofertas       → 18132681.0

N_PRODUTOS = len(P)
N_SEGMENTOS = len(S)
# ====================== MODEL ======================
model = ConcreteModel(name="Alocacao_Ofertas_Serasa")

# === Sets ===
model.P = Set(initialize=P)      # produtos
model.S = Set(initialize=S)  # segmentos

# === Parameters ===
# c[i,s] = taxa de conversão
model.c = Param(model.P, model.S, initialize=taxa_conversao_is)

# r[i] = ticket médio
model.r = Param(model.P, initialize=ticket_medio_i)

# n[s] = número de elegíveis no segmento
model.n = Param(model.S, initialize=usuarios_elegiveis_s)

# B = orçamento total de ofertas
model.B = Param(initialize=total_ofertas,within=NonNegativeIntegers)

# === Variáveis de decisão ===
# x[i,s] = número de ofertas do produto i para o segmento s
model.x = Var(model.P, model.S, domain=NonNegativeIntegers)

# === Função Objetivo ===
def objetivo_rule(model):
    return sum(model.c[i, s] * model.r[i] * model.x[i, s] 
               for i in model.P 
               for s in model.S)

model.objetivo = Objective(rule=objetivo_rule, sense=maximize)

# === Restrições ===

# (1) Limite de ofertas por segmento
def elegiveis_limite_rule(model, s):
    return sum(model.x[i, s] for i in model.P) <= model.n[s]

model.elegiveis_limite = Constraint(model.S, rule=elegiveis_limite_rule)

# (2) Limite total de ofertas
def oferta_limite_rule(model):
    return sum(model.x[i, s] for i in model.P for s in model.S) == model.B

model.oferta_limite = Constraint(rule=oferta_limite_rule)

# (3) Cada produto recebe no mínimo uma fração justa do orçamento
#     Se são 8 produtos, o "justo" seria 12.5% cada.
#     Usamos metade disso como piso → ~6.25%
def min_por_produto_rule(model, i):
    piso = (1.0 / N_PRODUTOS) * 0.5  # metade da fração uniforme
    return sum(model.x[i, s] for s in model.S) >= piso * model.B

model.min_prod = Constraint(model.P, rule=min_por_produto_rule)

# (4) Nenhum produto domina mais que X% do orçamento
def max_por_produto_rule(model, i):
    teto = min(3.0 / N_PRODUTOS, 0.35)  # 3x a fração uniforme, máx 35%
    return sum(model.x[i, s] for s in model.S) <= teto * model.B

model.max_prod = Constraint(model.P, rule=max_por_produto_rule)

# (5) Cada segmento recebe no mínimo uma fração proporcional ao seu tamanho
#     Garante que segmentos grandes não fiquem com 1 oferta
def min_por_segmento_rule(model, s):
    # pelo menos 10% dos elegíveis daquele segmento recebem alguma oferta
    return sum(model.x[i, s] for i in model.P) >= 0.10 * model.n[s]

model.min_seg = Constraint(model.S, rule=min_por_segmento_rule)

# ====================== RESOLVER ======================
solver = SolverFactory('cplex')
results = solver.solve(model, tee=True)    # tee=True mostra o log

print("\n=== RESULTADO ===")
print("Status:", results.solver.status)
print("Termination condition:", results.solver.termination_condition)
print("Receita esperada máxima:", value(model.objetivo))

# ====================== SOLUÇÃO EM DATAFRAME ======================
sol = []
for i in model.P:
    for s in model.S:
        _x = value(model.x[i, s])
        _c = value(model.c[i, s])
        _r = value(model.r[i])
        if _x > 0:
            sol.append({
                'sk_produto': i,
                'sk_consumidor': s,
                'possui_oferta*': round(_x),
                'conversao_efetiva*': round(_x * _c),
                'receita_gerada*': round(_x * _c * _r),
            })

df_solucao = pd.DataFrame(sol)
print(df_solucao.head(10))

analise = df_solucao.agg({'possui_oferta*': 'sum', 'conversao_efetiva*': 'sum', 'receita_gerada*': 'sum'})
possui_oferta_opt = analise.values[0]
conversao_efetiva_opt = analise.values[1]

print("=== ANÁLISE DA SOLUÇÃO OTIMIZADA ===")
print(f"shape: {df_solucao.shape[0]:,}")
print(f"Total de ofertas: {possui_oferta_opt:0,}")
print(f"Total de conversões efetivas: {conversao_efetiva_opt:0,}")
print(f'Taxa de conversão Otimizada: {(conversao_efetiva_opt/possui_oferta_opt)*100:.2f}%')
print(f'Receita Gerada Otimizada: {analise.values[2]:0,.2f}')

#df_solucao.to_csv("solucao_otimizacao_o.csv", index=False)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.2.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2024.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\eduar\AppData\Local\Temp\tmp70s_tlsq.cplex.log' open.
CPLEX> Problem 'C:\Users\eduar\AppData\Local\Temp\tmp6pa4c1p5.pyomo.lp' read.
Read time = 0.02 sec. (0.97 ticks)
CPLEX> Problem name         : C:\Users\eduar\AppData\Local\Temp\tmp6pa4c1p5.pyomo.lp
Objective sense      : Maximize
Variables            :    9205  [General Integer: 9205]
Objective nonzeros   :    9205
Linear constraints   :    2645  [Less: 1322,  Greater: 1322,  Equal: 1]
  Nonzeros           :   46025
  RHS nonzeros       :    2645

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective non

In [346]:
df_solucao.to_parquet(gold_dir / "solucao_otimizacao.parquet", index=False)